In [ ]:
!pip install -q llama-index llama-index-llms-gemini pymupdf
!pip install -q llama-index-embeddings-huggingface
!pip install -q nest_asyncio
!pip install -q llama-index-retrievers-bm25
!pip install -q sentence-transformers

!pip install llama-index-llms-anthropic

from llama_index.llms.anthropic import Anthropic



import os
import fitz  # PyMuPDF
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
import nest_asyncio
from llama_index.core import Settings

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 101.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 115.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.5/164.5 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 11.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cpu requires setuptools<82, but you have setuptools 83.0.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install -q llama-index llama-index-llms-anthropic pymupdf
!pip install -q llama-index-embeddings-huggingface
!pip install -q nest_asyncio
!pip install -q llama-index-retrievers-bm25
!pip install -q sentence-transformers

import os
import fitz
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
import nest_asyncio
from llama_index.core import Settings, VectorStoreIndex
from llama_index.llms.anthropic import Anthropic
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [ ]:


nest_asyncio.apply()

os.environ["ANTHROPIC_API_KEY"] = "Insert API Key"
llm = Anthropic(model="claude-haiku-4-5", api_key=os.environ["ANTHROPIC_API_KEY"])
Settings.llm = llm

def process_and_index_pdf(pdf_path):
    documents = load_pdf_with_pymupdf(pdf_path)
    vector_index = VectorStoreIndex.from_documents(documents)
    print(f"Indexed {len(documents)} document chunks")
    return vector_index



In [ ]:
from google.colab import files
import os

def upload_pdf():
    """Upload a PDF file and return its path."""
    print("Please select a PDF file to upload:")
    uploaded = files.upload()

    for filename in uploaded.keys():
        if filename.endswith('.pdf'):
            # Save to the sample_docs directory
            pdf_path = os.path.join("sample_docs", filename)

            # Create directory if it doesn't exist
            os.makedirs("sample_docs", exist_ok=True)

            # Save the file
            with open(pdf_path, 'wb') as f:
                f.write(uploaded[filename])

            print(f"PDF saved to {pdf_path}")
            return pdf_path
        else:
            print(f"File {filename} is not a PDF. Please upload a PDF file.")

    return None

In [ ]:
pdf_path = upload_pdf()

Please select a PDF file to upload:


Saving sample-sdf-document.pdf to sample-sdf-document.pdf
PDF saved to sample_docs/sample-sdf-document.pdf


In [ ]:
def extract_text_from_pdf(pdf_path):
    """Extract text from a PDF file using PyMuPDF."""
    doc = fitz.open(pdf_path)

    # Extract text from all pages
    text = "\n".join([page.get_text() for page in doc])

    # Print some stats
    print(f"PDF: {pdf_path}")
    print(f"Number of pages: {len(doc)}")
    print(f"Extracted {len(text.split())} words from the PDF.")

    # Close the document
    doc.close()

    return text

In [ ]:
from llama_index.core import Document
from typing import List

def load_pdf_with_pymupdf(pdf_path: str) -> List[Document]:
    """Load a PDF and convert it to LlamaIndex Document format using PyMuPDF."""
    # Open the PDF
    doc = fitz.open(pdf_path)

    # Extract text from each page
    documents = []

    for i, page in enumerate(doc):
        text = page.get_text()

        # Skip empty pages
        if not text.strip():
            continue

        # Create Document object with metadata
        documents.append(
            Document(
                text=text,
                metadata={
                    "file_name": os.path.basename(pdf_path),
                    "page_number": i + 1,
                    "total_pages": len(doc)
                }
            )
        )

    # Close the document
    doc.close()

    # Print stats
    print(f"Processed {pdf_path}:")
    print(f"Extracted {len(documents)} pages with content")

    return documents

In [ ]:
pdf_docs = load_pdf_with_pymupdf(pdf_path)

Processed sample_docs/sample-sdf-document.pdf:
Extracted 3 pages with content


In [ ]:
embed_model = HuggingFaceEmbedding(model_name="intfloat/e5-small-v2")
Settings.embed_model = embed_model

index = process_and_index_pdf(pdf_path)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Processed sample_docs/sample-sdf-document.pdf:
Extracted 3 pages with content
Indexed 3 document chunks


In [ ]:
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Semantic chunking needs an embedding model to detect topic shifts
# embed_model = HuggingFaceEmbedding(model_name="intfloat/e5-small-v2")
semantic_splitter = SemanticSplitterNodeParser(embed_model=embed_model)
chunks_semantic = semantic_splitter.get_nodes_from_documents(pdf_docs)
print(f"Total Semantic Chunks Created: {len(chunks_semantic)}")


for i, chunk in enumerate(chunks_semantic):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk.text)

Total Semantic Chunks Created: 6

--- Chunk 1 ---
Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: ÄKTATM ready Flow Kit Storage Conditions
To Whom It May Concern,
The recommended storage temperature for standard ÄKTA ready flow kits is provided in Section 8.3 of the Operating 
Instructions 28960345 and specified as > +5 C. This recommendation also applies to all modified ÄKTA ready flow kits 
as well, including the two listed in the below table. Extended storage below the recommended +5
could lead to 
brittleness or cracking of the plastic connectors. 

--- Chunk 2 ---
However, the operating temperature of ÄKTA ready flow kits is +2 C to 
+40 C. If the kits are allowed to acclimate to a warmer temperature before being used this would reduce the risk of
damage to the kit during setup and handling.
Description
Part Number
Operating Temperature
High Flow Kit F, Modified, ÄKTA ready
29477427
+2 C to +40 C
High Flow Gradient C, Modified, ÄKT

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Closed Source LLM (antrophic)
llm = Anthropic(model="claude-haiku-4-5", api_key=os.environ["ANTHROPIC_API_KEY"])
Settings.llm = llm

# # Initialize embedding model
# embed_model = HuggingFaceEmbedding(model_name="intfloat/e5-small-v2")
# Settings.embed_model = embed_model

def process_and_index_pdf(pdf_path):
    """Process a PDF and create both vector and keyword indices."""
    # Load documents
    documents = load_pdf_with_pymupdf(pdf_path)

    # Create vector index
    vector_index = VectorStoreIndex.from_documents(documents)

    print(f"Indexed {len(documents)} document chunks")

    return vector_index

In [ ]:
def expand_query(query: str, num_expansions: int = 3) -> list:
    """Expand a query to include related terms using Gemini."""
    prompt = f"""
    I need to search a pharmaceutical quality document with this query: "{query}"

    Please help me expand this query by generating {num_expansions} alternative versions that:
    1. Use different but related terminology
    2. Include relevant pharmaceutical/quality terms that might appear in a certificate or SDF
    3. Cover similar concepts but phrased differently

    Format your response as a list of alternative queries only, with no additional text.
    """

    response = llm.complete(prompt)

    # Extract the expanded queries
    expanded_queries = [line.strip() for line in response.text.split('\n') if line.strip()]

    # Add the original query if needed
    if query not in expanded_queries:
        expanded_queries = [query] + expanded_queries

    return expanded_queries

query expansion test

In [ ]:
expanded = expand_query("What test methods were used for quality control?")
for i, q in enumerate(expanded):
    print(f"{i+1}. {q}")

1. What test methods were used for quality control?
2. 1. "Which analytical procedures and testing techniques were employed for quality assurance?"
3. 2. "What are the specification test methods and acceptance criteria used in quality control testing?"
4. 3. "Which analytical methods and test protocols were applied to verify product quality and compliance?"


In [ ]:
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import QueryFusionRetriever

# Function to create a query engine that uses query expansion
def create_query_expansion_engine(index):
    """Create a query engine that uses query expansion."""
    # First create multiple retrievers (base retriever)
    base_retriever = index.as_retriever(similarity_top_k=2)

    # Create a query fusion retriever
    fusion_retriever = QueryFusionRetriever(
        retrievers=[base_retriever],
        llm=llm,
        similarity_top_k=2,
        num_queries=3,  # Generate 3 queries per original query
        mode="reciprocal_rerank"  # Use reciprocal rank fusion
    )

    # Create the query engine with the fusion retriever
    query_engine = RetrieverQueryEngine.from_args(
        retriever=fusion_retriever,
        llm=llm,
        verbose=True
    )

    return query_engine

In [ ]:
expanded_query_engine = create_query_expansion_engine(index)
response = expanded_query_engine.query("What test methods were used for quality control?")
print(response)

Based on the quality control documentation provided, the following test methods were used:

1. **Autoclave - Pump tubing**: 121°C for greater than 15 minutes
2. **Gamma Irradiation - Inlets**: 25.0 - 40.0 kGy
3. **Flow Rate Test**: Per SOP-QC-042
4. **Pressure Integrity**: Maximum 0.5 bar with a 5-minute hold
5. **Visual Inspection**: Checking for visible defects
6. **Package Integrity**: Verification that packaging is sealed with no damage

Additionally, regulatory conformance testing was performed to ensure that all polymeric materials in contact with process fluids comply with United States Pharmacopeia (USP) <88> Biological Reactivity Test Class VI standards.


In [ ]:
from llama_index.core import VectorStoreIndex
from llama_index.retrievers.bm25 import BM25Retriever

def create_hybrid_retriever(index, query, top_k=2):
  # vector retrival
    vector_retriever = index.as_retriever(similarity_top_k=top_k)
    vector_nodes = vector_retriever.retrieve(query)

# using keywords
    nodes = [node for node in index.docstore.docs.values()]
    bm25_retriever = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=top_k)
    keyword_nodes = bm25_retriever.retrieve(query)

    k = 60
    rrf_scores = {}

    for rank, node in enumerate(vector_nodes):
        nid = node.node_id
        rrf_scores[nid] = rrf_scores.get(nid, 0) + 1 / (k + rank + 1)

    for rank, node in enumerate(keyword_nodes):
        nid = node.node_id
        rrf_scores[nid] = rrf_scores.get(nid, 0) + 1 / (k + rank + 1)

    max_score = max(rrf_scores.values()) if rrf_scores else 1
    normalized_scores = {nid: score / max_score for nid, score in rrf_scores.items()}

    all_nodes = {node.node_id: node for node in vector_nodes + keyword_nodes}

    sorted_nodes = sorted(
        all_nodes.values(),
        key=lambda x: normalized_scores.get(x.node_id, 0),
        reverse=True
    )[:top_k]

    # Apply normalized score back to node
    for node in sorted_nodes:
        node.score = normalized_scores.get(node.node_id, 0)

    return sorted_nodes


In [ ]:
hybrid_nodes = create_hybrid_retriever(index, "What are the storage conditions for this product?")
for i, node in enumerate(hybrid_nodes):
    print(f"Result {i+1} (Score: {node.score:.4f}):")
    print(node.get_text())
    print("-" * 40)
    print("")


DEBUG:bm25s:Building index from IDs objects


Result 1 (Score: 1.0000):
Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: ÄKTATM ready Flow Kit Storage Conditions
To Whom It May Concern,
The recommended storage temperature for standard ÄKTA ready flow kits is provided in Section 8.3 of the Operating 
Instructions 28960345 and specified as > +5 C. This recommendation also applies to all modified ÄKTA ready flow kits 
as well, including the two listed in the below table. Extended storage below the recommended +5
could lead to 
brittleness or cracking of the plastic connectors. However, the operating temperature of ÄKTA ready flow kits is +2 C to 
+40 C. If the kits are allowed to acclimate to a warmer temperature before being used this would reduce the risk of
damage to the kit during setup and handling.
Description
Part Number
Operating Temperature
High Flow Kit F, Modified, ÄKTA ready
29477427
+2 C to +40 C
High Flow Gradient C, Modified, ÄKTA ready
29184612
+2 C to +40 C
Operating I

In [ ]:
# reranking

from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.core.schema import NodeWithScore

# Create a reranker
def rerank_results(nodes, query, top_n=2):
    """Rerank retrieved nodes using the Sentence Transformer reranker."""
    # Create the reranker
    reranker = SentenceTransformerRerank(
        model="cross-encoder/ms-marco-MiniLM-L-6-v2",
        top_n=top_n
    )

    # Rerank the nodes
    reranked_nodes = reranker.postprocess_nodes(
        nodes,
        query_str=query
    )

    return reranked_nodes

# Function to demonstrate the reranking process
def demonstrate_reranking(index, query, top_k=3):
    """Demonstrate the reranking process on retrieval results."""
    # First retrieve more nodes than we need
    retriever = index.as_retriever(similarity_top_k=top_k)
    nodes = retriever.retrieve(query)

    print(f"Query: {query}")
    print("\nOriginal Retrieval Order:")
    for i, node in enumerate(nodes):
        print(f"{i+1}. (Score: {node.score:.4f}) - {node.get_text()[:100]}...")

    # Now rerank them
    reranked_nodes = rerank_results(nodes, query, top_n=2)

    print("\nAfter Reranking:")
    for i, node in enumerate(reranked_nodes):
        print(f"{i+1}. (Score: {node.score:.4f}) - {node.get_text()[:100]}...")

    # Create comparison dataframe
    results = []

    # Original ranking
    for i, node in enumerate(nodes):
        results.append({
            "Stage": "Original Retrieval",
            "Rank": i + 1,
            "Score": node.score,
            "Content": node.get_text()[:150] + "...",
            "Page": node.metadata.get("page_number", "Unknown")
        })

    # Reranked
    for i, node in enumerate(reranked_nodes):
        results.append({
            "Stage": "After Reranking",
            "Rank": i + 1,
            "Score": node.score,
            "Content": node.get_text()[:150] + "...",
            "Page": node.metadata.get("page_number", "Unknown")
        })

    results_df = pd.DataFrame(results)
    display(results_df)

    return results_df

# Example usage:
reranking_demo = demonstrate_reranking(index, "What sterilization method was used?", top_k=3)

Query: What sterilization method was used?

Original Retrieval Order:
1. (Score: 0.7800) - Cytiva
cytiva.com
Certificate of Quality
This product is manufactured in compliance with our ISO 900...
2. (Score: 0.7743) - Certificate of Quality 
This product is manufactured in compliance with our ISO 9001 certified quali...
3. (Score: 0.7617) - Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: ÄK...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


After Reranking:
1. (Score: -11.1406) - Cytiva
cytiva.com
Certificate of Quality
This product is manufactured in compliance with our ISO 900...
2. (Score: -11.2391) - Certificate of Quality 
This product is manufactured in compliance with our ISO 9001 certified quali...


,Stage,Rank,Score,Content,Page
0,Original Retrieval,1,-11.140560,Cytiva\ncytiva.com\nCertificate of Quality\nTh...,3
1,Original Retrieval,2,-11.239133,Certificate of Quality \nThis product is manuf...,2
2,Original Retrieval,3,-11.253381,"Cytiva\n100 Results Way\nMarlborough, MA 01752...",1
3,After Reranking,1,-11.140560,Cytiva\ncytiva.com\nCertificate of Quality\nTh...,3
4,After Reranking,2,-11.239133,Certificate of Quality \nThis product is manuf...,2


------------------------------------------------------------
Final Pipeline

In [ ]:
# rag pipeline

from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.retrievers import BaseRetriever
from llama_index.core.schema import NodeWithScore, QueryBundle

def build_rag_pipeline(index):
    """Build a simple but effective RAG pipeline with hybrid retrieval and reranking."""

    # Get all nodes from the index's docstore
    nodes = list(index.docstore.docs.values())

    # Determine safe top_k value (number of nodes to retrieve)
    # Must be at least 1 and no more than the number of available nodes
    num_nodes = len(nodes)
    safe_top_k = min(2, max(1, num_nodes))

    print(f"Index contains {num_nodes} nodes, using top_k={safe_top_k}")

    # Step 1: Create a hybrid retriever combining vector and keyword search
    # First, get the vector retriever (for semantic understanding)
    vector_retriever = index.as_retriever(
        similarity_top_k=safe_top_k  # Retrieve top 2 most similar chunks
    )

    # Next, create a BM25 retriever (for keyword matching)
    bm25_retriever = BM25Retriever.from_defaults(
        nodes=nodes,
        similarity_top_k=safe_top_k  # Retrieve top 2 most similar chunks
    )

    # Create a proper hybrid retriever class
    class HybridRetriever(BaseRetriever):
        """Hybrid retriever that combines vector and keyword search results."""

        def __init__(self, vector_retriever, keyword_retriever, top_k=2):
            """Initialize with vector and keyword retrievers."""
            self.vector_retriever = vector_retriever
            self.keyword_retriever = keyword_retriever
            self.top_k = top_k
            super().__init__()

        def _retrieve(self, query_bundle, **kwargs):
            """Retrieve from both retrievers and combine results."""
            # Get results from both retrievers
            vector_nodes = self.vector_retriever.retrieve(query_bundle)
            keyword_nodes = self.keyword_retriever.retrieve(query_bundle)

            # Combine all nodes
            all_nodes = list(vector_nodes) + list(keyword_nodes)

            # Remove duplicates (by node_id)
            unique_nodes = {}
            for node in all_nodes:
                if node.node_id not in unique_nodes:
                    unique_nodes[node.node_id] = node

            # Sort by score (higher is better)
            sorted_nodes = sorted(
                unique_nodes.values(),
                key=lambda x: x.score if hasattr(x, 'score') else 0.0,
                reverse=True
            )

            return sorted_nodes[:self.top_k]  # Return top results

    # Create our hybrid retriever instance
    hybrid_retriever = HybridRetriever(
        vector_retriever=vector_retriever,
        keyword_retriever=bm25_retriever,
        top_k=safe_top_k
    )

    # Step 2: Create a reranker to prioritize the most relevant chunks
    if num_nodes > 1:
        reranker = SentenceTransformerRerank(
            model="cross-encoder/ms-marco-MiniLM-L-6-v2",
            top_n=min(2, num_nodes)  # Keep only top 2 results after reranking
        )
        node_postprocessors = [reranker]
    else:
        node_postprocessors = []


    # Step 3: Build the query engine
    query_engine = RetrieverQueryEngine.from_args(
        retriever=hybrid_retriever,
        llm=llm,
        node_postprocessors=node_postprocessors
    )

    return query_engine

In [ ]:
index = process_and_index_pdf(pdf_path)
rag_engine = build_rag_pipeline(index)
response = rag_engine.query("What are the storage conditions specified in the certificate?")
print('\nFinal Response:\n ---------------------- \n')
print(response)

Processed sample_docs/sample-sdf-document.pdf:
Extracted 3 pages with content


DEBUG:bm25s:Building index from IDs objects


Indexed 3 document chunks
Index contains 3 nodes, using top_k=2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


Final Response:
 ---------------------- 

The certificate of quality document does not specify storage conditions. The storage conditions are detailed in a separate letter from the Product Manager, which states that the recommended storage temperature for ÄKTA ready flow kits is greater than +5°C. Extended storage below this recommended temperature could lead to brittleness or cracking of the plastic connectors. However, the operating temperature range for these kits is +2°C to +40°C, and allowing the kits to acclimate to a warmer temperature before use would reduce the risk of damage during setup and handling.
